In [29]:
!pip install transformers datasets torch scikit-learn

In [30]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [31]:
reviews_df = pd.read_csv("data/output/processed/reviews.csv")

print(reviews_df.head())
print("Shape:", reviews_df.shape)
print("Columns:", reviews_df.columns.tolist())
print("\nSentiment distribution:")
print(reviews_df['sentiment_label'].value_counts())

  review_id meal_id            meal_name      restaurant  rating  \
0   R000001   M0001  Classic Beef Burger  Mama's Kitchen     1.9   
1   R000002   M0001  Classic Beef Burger  Mama's Kitchen     3.8   
2   R000003   M0001  Classic Beef Burger  Mama's Kitchen     4.0   
3   R000004   M0001  Classic Beef Burger  Mama's Kitchen     4.7   
4   R000005   M0001  Classic Beef Burger  Mama's Kitchen     4.0   

                                         review_text sentiment_label  \
0  Disappointed with the quality. Not what I expe...        negative   
1  Pretty good, I've had better but wouldn't say ...         neutral   
2  Super fast delivery and the food was outstandi...        positive   
3  Perfect seasoning and amazing presentation. Wo...        positive   
4  Loved every bite. The packaging was also excel...        positive   

  review_date  helpful_votes  
0  2026-03-19              8  
1  2026-04-05             44  
2  2026-02-01             50  
3  2025-12-14             10  
4  

In [32]:
label_map = {"negative": 0, "neutral": 1, "positive": 2}
reviews_df['label'] = reviews_df['sentiment_label'].map(label_map)

print("Label distribution:")
print(reviews_df['label'].value_counts())

Label distribution:
label
2    1070
0     394
1     371
Name: count, dtype: int64


In [33]:
import random

def add_noise(label, noise_rate=0.1):
    """Randomly flip labels to simulate real-world noise."""
    if random.random() < noise_rate:
        choices = [0, 1, 2]
        choices.remove(label)
        return random.choice(choices)
    return label

random.seed(42)
reviews_df['label'] = reviews_df['label'].apply(add_noise)

print("Label distribution after noise:")
print(reviews_df['label'].value_counts())

Label distribution after noise:
label
2    1007
0     420
1     408
Name: count, dtype: int64


In [34]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    reviews_df['review_text'].tolist(),
    reviews_df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

print(f"Train size: {len(train_texts)} | Test size: {len(test_texts)}")

Train size: 1468 | Test size: 367


In [35]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization complete.")
print("Keys in encoding:", list(train_encodings.keys()))

Tokenization complete.
Keys in encoding: ['input_ids', 'token_type_ids', 'attention_mask']


In [36]:
class ReviewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ReviewsDataset(train_encodings, train_labels)
test_dataset = ReviewsDataset(test_encodings, test_labels)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Test dataset:  {len(test_dataset)} samples")

Train dataset: 1468 samples
Test dataset:  367 samples


In [37]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [38]:
def compute_metrics(eval_pred):
    """Compute accuracy for the Trainer."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    weight_decay=0.01,          
    warmup_ratio=0.1,           
    learning_rate=2e-5,         
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [40]:
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [41]:
trainer.train()

c:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.961172,0.421412,0.904632
2,0.435804,0.390900,0.904632
3,0.421459,0.396079,0.904632


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=276, training_loss=0.5485938528309697, metrics={'train_runtime': 990.3698, 'train_samples_per_second': 4.447, 'train_steps_per_second': 0.279, 'total_flos': 52053289665048.0, 'train_loss': 0.5485938528309697, 'epoch': 3.0})

In [42]:
eval_results = trainer.evaluate()
print("Evaluation Results:")
for k, v in eval_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

c:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.421459,0.421163,3,0.904632


Evaluation Results:
  eval_loss: 0.4212
  eval_accuracy: 0.9046


In [ ]:
predictions_output = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions_output.predictions, axis=-1)

label_names = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
print("Classification Report:")
print(classification_report(test_labels, pred_labels, labels=[0, 1, 2], target_names=label_names))

c:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Classification Report:
              precision    recall  f1-score   support

    NEGATIVE       0.92      0.89      0.91        75
     NEUTRAL       0.92      0.79      0.85        90
    POSITIVE       0.89      0.96      0.93       202

    accuracy                           0.90       367
   macro avg       0.91      0.88      0.89       367
weighted avg       0.91      0.90      0.90       367



## 12. Apply to All Reviews & Save Output

In [44]:
label_map = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
score_map = {0: -1, 1: 0, 2: 1}

all_encodings = tokenizer(
    reviews_df['review_text'].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)
all_dataset = ReviewsDataset(all_encodings, [0] * len(reviews_df))  # dummy labels

all_preds = trainer.predict(all_dataset)
pred_indices = np.argmax(all_preds.predictions, axis=-1)

reviews_df['sentiment'] = [label_map[p] for p in pred_indices]
reviews_df['sentiment_score'] = [score_map[p] for p in pred_indices]

print("Sentiment distribution in full dataset:")
print(reviews_df['sentiment'].value_counts())

c:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Sentiment distribution in full dataset:
sentiment
POSITIVE    1070
NEGATIVE     394
NEUTRAL      371
Name: count, dtype: int64


In [46]:
output_df = reviews_df[
    ['review_id', 'meal_id', 'review_text', 'sentiment', 'sentiment_score']
]

output_df.to_csv("data/output/processed/reviews_with_sentiment.csv", index=False)

print("Saved to: data/output/processed/reviews_with_sentiment.csv")
print(output_df.head())

Saved to: data/output/processed/reviews_with_sentiment.csv
  review_id meal_id                                        review_text  \
0   R000001   M0001  Disappointed with the quality. Not what I expe...   
1   R000002   M0001  Pretty good, I've had better but wouldn't say ...   
2   R000003   M0001  Super fast delivery and the food was outstandi...   
3   R000004   M0001  Perfect seasoning and amazing presentation. Wo...   
4   R000005   M0001  Loved every bite. The packaging was also excel...   

  sentiment  sentiment_score  
0  NEGATIVE               -1  
1   NEUTRAL                0  
2  POSITIVE                1  
3  POSITIVE                1  
4  POSITIVE                1  
